In [ ]:
!pip install lightgbm


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
import lightgbm as lgb

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

In [ ]:
y = train["Class"].map({"Good":0, "NG":1})
train = train.drop("Class", axis=1)


In [ ]:
# 모든 object 컬럼을 category→codes로 변환
for col in train.columns:
    if train[col].dtype == "object":
        all_vals = pd.concat([train[col], test[col]])
        codes = {v: i for i, v in enumerate(all_vals.unique())}
        train[col] = train[col].map(codes)
        test[col]  = test[col].map(codes)


In [ ]:
proc_cols = [c for c in train.columns if "Proc_" in c]
xy_cols   = [c for c in train.columns if c.startswith(("X","Y"))]
g_cols    = [c for c in train.columns if c.startswith("G")]

def add_features(df):
    df["proc_mean"] = df[proc_cols].mean(axis=1)
    df["proc_std"]  = df[proc_cols].std(axis=1)

    df["xy_mean"] = df[xy_cols].mean(axis=1)
    df["xy_std"]  = df[xy_cols].std(axis=1)

    df["g_mean"] = df[g_cols].mean(axis=1)
    df["g_std"]  = df[g_cols].std(axis=1)

add_features(train)
add_features(test)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42
)


In [ ]:
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.log_evaluation(period=50)]
)

[LightGBM] [Info] Number of positive: 87, number of negative: 489
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019165 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 152685
[LightGBM] [Info] Number of data points in the train set: 576, number of used features: 804
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.151042 -> initscore=-1.726454
[LightGBM] [Info] Start training from score -1.726454
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.03, n_estimators=500,
               num_leaves=63, random_state=42, subsample=0.8)

In [ ]:
val_prob = model.predict_proba(X_val)[:, 1]

def compute_profit(pred, true):
    good_correct = ((pred==0) & (true==0)).sum()
    ng_correct   = ((pred==1) & (true==1)).sum()
    false_block  = ((pred==1) & (true==0)).sum()
    return good_correct*100 + ng_correct*20000 - false_block*20000

thr_list = np.linspace(0.01, 0.30, 100)
profits = []

for th in thr_list:
    pred = (val_prob >= th).astype(int)
    profits.append(compute_profit(pred, y_val))

best_thr = thr_list[np.argmax(profits)]
best_thr, max(profits)

(np.float64(0.28828282828282825), np.int64(-28400))

In [ ]:
# Store original IDs for submission
test_ids = test["ID"]

# Create a copy of the test data for preprocessing
test_preprocessed = test.copy()

# Drop the 'ID' column as it's not a feature for the model
test_preprocessed = test_preprocessed.drop(columns=["ID"], errors='ignore')

# Re-apply the categorical encoding logic
# This assumes 'train' and 'test' variables are the raw DataFrames from pd.read_csv.
# This part ensures consistency with how X_train was created if prior cells were not executed.

train_temp = train.drop("Class", axis=1).copy()

for col in train_temp.columns:
    if train_temp[col].dtype == "object":
        all_vals = pd.concat([train_temp[col], test_preprocessed[col]])
        codes = {v: i for i, v in enumerate(all_vals.unique())}
        train_temp[col] = train_temp[col].map(codes)
        test_preprocessed[col] = test_preprocessed[col].map(codes)

# Re-apply feature engineering
# Define proc_cols, xy_cols, g_cols locally if they might not be in scope or are outdated
proc_cols = [c for c in train_temp.columns if "Proc_" in c]
xy_cols   = [c for c in train_temp.columns if c.startswith(("X","Y"))]
g_cols    = [c for c in train_temp.columns if c.startswith("G")]

def add_features_local(df):
    if proc_cols and not df[proc_cols].empty:
        df["proc_mean"] = df[proc_cols].mean(axis=1)
        df["proc_std"]  = df[proc_cols].std(axis=1)

    if xy_cols and not df[xy_cols].empty:
        df["xy_mean"] = df[xy_cols].mean(axis=1)
        df["xy_std"]  = df[xy_cols].std(axis=1)

    if g_cols and not df[g_cols].empty:
        df["g_mean"] = df[g_cols].mean(axis=1)
        df["g_std"]  = df[g_cols].std(axis=1)

add_features_local(test_preprocessed)

# Ensure `test_preprocessed` columns match `X_train` columns
# X_train should be available from the kernel state and represents the features used for training.
final_test_features = test_preprocessed[X_train.columns]

test_proba = model.predict_proba(final_test_features)[:,1]
decision = test_proba >= best_thr

n = len(test_proba)

# ID 리스트 생성 (L 먼저, 그 다음 P)
ids = []
probs = []
decisions = []

# L 먼저 모두 넣기
for i in range(n):
    ids.append(f"{test_ids.iloc[i]}_L") # Use original test_ids
    probs.append(test_proba[i])
    decisions.append(decision[i])

# P 나중에 모두 넣기
for i in range(n):
    ids.append(f"{test_ids.iloc[i]}_P") # Use original test_ids
    probs.append(test_proba[i])
    decisions.append(decision[i])

submission = pd.DataFrame({
    "ID": ids,
    "probability": probs,
    "decision": decisions
})

submission.to_csv("submission.csv", index=False)
submission.head()


,ID,probability,decision
0,ID_0_L,0.209782,False
1,ID_1_L,0.000029,False
2,ID_2_L,0.000211,False
3,ID_3_L,0.000372,False
4,ID_4_L,0.002715,False


from matplotlib import pyplot as plt
_df_0['probability'].plot(kind='hist', bins=20, title='probability')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('ID').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['probability'].plot(kind='line', figsize=(8, 4), title='probability')
plt.gca().spines[['top', 'right']].set_visible(False)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_3['ID'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_3, x='probability', y='ID', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
true_count = (submission["decision"] == True).sum()
false_count = (submission["decision"] == False).sum()

print("True 개수:", true_count)
print("False 개수:", false_count)


True 개수: 88
False 개수: 844


In [ ]:
true_count = decision.sum()        # True는 1로 계산됨
false_count = len(decision) - true_count

print("True 개수:", true_count)
print("False 개수:", false_count)


True 개수: 44
False 개수: 422
